In [1]:
pip install ortools

^C
Note: you may need to restart the kernel to use updated packages.


In [1]:
import ortools

In [8]:
# Variables:
#   steps[i] = position of robot at step i (flattened index 0–24)
#
# Domains:
#   steps[i] ∈ {0,1,...,24}
#
# Constraints:
#   1. Start at (1,1)
#   2. End at (4,4)
#   3. Move only diagonally → (±1, ±1)
#   4. Avoid obstacles
#   5. Minimize number of steps (shortest path)

from ortools.sat.python import cp_model

model = cp_model.CpModel()

grid_size = 5
max_steps = 6  
def to_index(r, c):
    return r * grid_size + c

start = to_index(1, 1)   
target = to_index(4, 4)  

obstacles = [7, 11, 13]

steps = [model.NewIntVar(0, 24, f'step{i}') for i in range(max_steps)]

model.Add(steps[0] == start)

end_flags = []
for i in range(max_steps):
    b = model.NewBoolVar(f'end_at_{i}')
    model.Add(steps[i] == target).OnlyEnforceIf(b)
    model.Add(steps[i] != target).OnlyEnforceIf(b.Not())
    end_flags.append(b)

model.Add(sum(end_flags) == 1)

moves = [4, 6, -4, -6]

for i in range(max_steps - 1):
    allowed = []
    for m in moves:
        for pos in range(25):
            nxt = pos + m
            if 0 <= nxt < 25:
                allowed.append([pos, nxt])
    model.AddAllowedAssignments([steps[i], steps[i+1]], allowed)

for s in steps:
    for obs in obstacles:
        model.Add(s != obs)

model.Minimize(sum(end_flags[i] * i for i in range(max_steps)))

solver = cp_model.CpSolver()
status = solver.Solve(model)

if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    path = []
    for i in range(max_steps):
        val = solver.Value(steps[i])
        path.append(val)
        if val == target:
            break

    print("Path:", path)
    for p in path:
        print(f"-> ({p//5}, {p%5})")
else:
    print("No solution")

Path: [6, 12, 18, 24]
-> (1, 1)
-> (2, 2)
-> (3, 3)
-> (4, 4)


In [ ]:
# Variables:
#   X[i][j] = 1 if land, 0 if water
#
# Domains:
#   X[i][j] ∈ {0,1}
#
# Constraints:
#   1. Grid values are fixed
#   2. Each land cell contributes to perimeter if adjacent to water or boundary

from ortools.sat.python import cp_model

model = cp_model.CpModel()

grid = [
    [1, 1, 0, 0, 0],
    [1, 1, 0, 1, 1],
    [0, 0, 0, 1, 1],
    [0, 1, 0, 0, 0],
    [0, 1, 1, 0, 0]
]

rows = len(grid)
cols = len(grid[0])


X = {}
for i in range(rows):
    for j in range(cols):
        X[i, j] = model.NewIntVar(0, 1, f'X_{i}_{j}')
        model.Add(X[i, j] == grid[i][j]) 
edges = []

for i in range(rows):
    for j in range(cols):
        if grid[i][j] == 1:
            directions = [(0,1), (0,-1), (1,0), (-1,0)]
            
            for dr, dc in directions:
                ni, nj = i + dr, j + dc
                
                edge = model.NewIntVar(0, 1, f'e_{i}_{j}_{dr}_{dc}')
                
                if 0 <= ni < rows and 0 <= nj < cols:
                    
                    model.Add(edge == 1).OnlyEnforceIf(X[ni, nj].Not())
                    model.Add(edge == 0).OnlyEnforceIf(X[ni, nj])
                else:
                    model.Add(edge == 1)
                
                edges.append(edge)


perimeter = model.NewIntVar(0, 1000, 'perimeter')
model.Add(perimeter == sum(edges))

solver = cp_model.CpSolver()
solver.Solve(model)

print("Perimeter:", solver.Value(perimeter))

Perimeter: 24


In [ ]:
# Variables:
#   X[i] = city visited at position i
#
# Domains:
#   X[i] ∈ {0,1,...,9}
#
# Constraints:
#   1. AllDifferent(X)
#   2. Complete tour (return to start)

from ortools.sat.python import cp_model

model = cp_model.CpModel()

n = 10
D = [
    [0, 29, 20, 21, 16, 31, 100, 12, 4, 31],
    [29, 0, 15, 29, 28, 40, 72, 21, 29, 41],
    [20, 15, 0, 15, 14, 25, 81, 9, 23, 27],
    [21, 29, 15, 0, 4, 12, 92, 12, 25, 13],
    [16, 28, 14, 4, 0, 16, 94, 9, 20, 16],
    [31, 40, 25, 12, 16, 0, 95, 24, 36, 3],
    [100,72, 81, 92, 94, 95, 0, 90, 101, 99],
    [12, 21, 9, 12, 9, 24, 90, 0, 15, 25],
    [4, 29, 23, 25, 20, 36, 101,15, 0, 35],
    [31, 41, 27, 13, 16, 3, 99, 25, 35, 0]
]

X = [model.NewIntVar(0, n-1, f'X{i}') for i in range(n)]

model.AddAllDifferent(X)

dist_vars = []
for i in range(n):
    d = model.NewIntVar(0, 200, f'd{i}')
    dist_vars.append(d)

for i in range(n-1):
    model.AddElement(X[i] * n + X[i+1],
                     [D[r][c] for r in range(n) for c in range(n)],
                     dist_vars[i])

model.AddElement(X[n-1] * n + X[0],
                 [D[r][c] for r in range(n) for c in range(n)],
                 dist_vars[n-1])

total_cost = model.NewIntVar(0, 2000, 'total_cost')
model.Add(total_cost == sum(dist_vars))

model.Minimize(total_cost)


solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 10

status = solver.Solve(model)

if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    route = [solver.Value(x) for x in X]
    print("Route:", route)
    print("Total Distance:", solver.Value(total_cost))
else:
    print("No solution found")

Route: [0, 7, 2, 1, 6, 5, 9, 3, 4, 8]
Total Distance: 247
